# Atividade: Nonograma

Nesta atividade, vamos aprender como codificar o quebra-cabeça **Nonograma** como uma instância SAT e resolvê-lo utilizando o PycoSAT, seguindo a mesma metodologia usada para o Sudoku.

## O problema do Nonograma

O Nonograma é um puzzle de lógica combinatória sobre uma grade bidimensional de dimensões m×n, onde o objetivo é determinar o estado binário de cada célula (preenchida ou vazia). As restrições do problema são dadas por sequências de inteiros associadas a cada linha e coluna, as quais especificam os comprimentos e a ordem exata de blocos contíguos de células preenchidas. Formalmente, o problema de decisão do Nonograma consiste em determinar se existe ao menos uma matriz binária que satisfaça simultaneamente todas as restrições de consistência local das linhas e das colunas.

Embora instâncias pequenas e comerciais de Nonogramas sejam projetadas para serem resolvidas através de deduções locais simples, a versão generalizada do problema é classificada como NP-difícil (NP-Hard). A prova de sua intratabilidade computacional foi originalmente estabelecida por Ueda e Nagao através de uma redução polinomial a partir do problema Circuit-SAT, que é sabidamente NP-completo.

## Esboço da prova de NP-Hardness via redução polinomial

A demonstração consiste em construir um mapeamento de tempo polinomial que transforma qualquer circuito booleano arbitrário (composto por variáveis de entrada e portas lógicas) em uma instância equivalente de Nonograma. Para isso, a grade do Nonograma é tratada como um plano de circuitos digitais, onde estruturas geométricas específicas (denominadas *gadgets*) simulam o comportamento dos componentes eletrônicos:

- **Gadget de Fio (Wire)**: fios são representados por sequências longas de células dispostas em diagonais ou linhas direcionadas. A paridade do deslocamento dos blocos preenchidos propaga o valor lógico através da grade. Uma configuração representa o valor lógico Verdadeiro (1) e a configuração alternativa representa o Falso (0). As pistas numéricas adjacentes forçam a consistência desse sinal ao longo de todo o percurso.
- **Gadget de Divisão (Splitter)**: para permitir que o sinal de uma única variável seja utilizado como entrada em múltiplas portas lógicas, utiliza-se um componente de ramificação. Esse gadget garante, por meio de restrições de colunas compartilhadas, que a configuração de preenchimento do fio de entrada seja replicada identicamente para dois ou mais fios de saída.
- **Gadget de Portas Lógicas (AND / OR / NOT)**: as operações booleanas são simuladas na interseção de fios horizontais e verticais. Por exemplo, em uma porta lógica AND, o gadget é projetado de forma que o preenchimento de uma célula crítica de saída só seja geometricamente possível se as pistas das linhas e colunas que representam as duas entradas estiverem ativadas no estado Verdadeiro. Qualquer outra combinação força a linha de saída a assumir o estado Falso.
- **Gadget de Cruzamento (Crossover)**: como os circuitos lógicos podem exigir o cruzamento de conexões e a grade do Nonograma é estritamente bidimensional, a prova introduz um gadget de cruzamento planar. Essa estrutura permite que dois fios independentes se cruzem perpendicularmente sem que o estado lógico de um interfira ou corrompa o estado lógico do outro.

Ao interconectar esses componentes, uma fórmula booleana complexa é inteiramente traduzida em um tabuleiro. A restrição final é aplicada à célula que representa a saída do circuito, forçando-a a ser Verdadeira.

Como a construção dessa grade consome tempo polinomial em relação ao tamanho do circuito original, e dado que o Nonograma resultante possui uma solução válida se, e somente se, o circuito lógico original for satisfatível, conclui-se que o problema do Nonograma é tão difícil quanto o Circuit-SAT. Portanto, o problema é formalmente NP-difícil, justificando o uso de abordagens robustas como o mapeamento para uma CNF (Conjunctive Normal Form) e a subsequente resolução via SAT solvers, conforme implementado a seguir.

Nas seções seguintes, o solucionador é construído passo a passo: cada célula de código é precedida por uma célula de texto explicando a ideia por trás do trecho implementado, no mesmo formato usado na atividade de Sudoku.

## Preparando o ambiente

A maioria dos solucionadores SAT, incluindo o PycoSAT, aceita apenas fórmulas na Forma Normal Conjuntiva (CNF), representadas em Python como listas de cláusulas, onde cada cláusula é uma lista de literais (inteiros não nulos, positivos ou negados). Vamos instalar e importar o PycoSAT antes de começar.

In [50]:
%pip install pycosat
import pycosat

### Representando um Nonograma em Python

Vamos representar um Nonograma como um dicionário com duas chaves:

- `"linhas"`: uma lista com uma entrada por linha da grade; cada entrada é a lista de inteiros com o comprimento de cada bloco contíguo de células preenchidas naquela linha, na ordem em que aparecem da esquerda para a direita. Uma linha sem blocos preenchidos é representada por uma lista vazia `[]`.
- `"colunas"`: o mesmo, mas uma entrada por coluna, na ordem de cima para baixo.

Por exemplo, o Nonograma 5×5 abaixo (`nonograma_simples`) tem, na primeira linha, dois blocos de tamanho 1 (`[1, 1]`), e assim por diante.

In [51]:
from typing import TypedDict

class Nonograma(TypedDict):
    linhas: list[list[int]]
    colunas: list[list[int]]

nonograma_simples: Nonograma = {
    "linhas": [
        [1, 1],
        [1, 1, 1],
        [1, 1],
        [1, 1],
        [1],
    ],
    "colunas": [
        [2],
        [1, 1],
        [1, 1],
        [1, 1],
        [2],
    ]
}

## Construindo um resolvedor para Nonograma

Assim como fizemos para o Sudoku, vamos codificar o Nonograma como uma fórmula CNF, resolvê-la com o PycoSAT e depois decodificar a atribuição satisfatória de volta para uma grade preenchida.

O roteiro é:

1. Escolher variáveis booleanas que representem o estado de cada célula da grade.
2. Para cada linha e cada coluna, enumerar todas as configurações de preenchimento que satisfazem as suas restrições de blocos.
3. Codificar, via CNF, que **exatamente uma** dessas configurações deve ocorrer em cada linha/coluna — usando uma transformação de Tseitin para evitar uma explosão exponencial de cláusulas.
4. Resolver a fórmula resultante com o PycoSAT.
5. Traduzir a atribuição satisfatória de volta para uma grade de células preenchidas/vazias.

Vamos criar uma classe `ResolvedorNonograma` e, célula a célula, implementar cada um desses passos.

In [52]:
class ResolvedorNonograma:
    pass

### Construtor

O construtor recebe um `Nonograma` e guarda o essencial para montar a CNF mais tarde:

- `m`, `n`: número de linhas e colunas da grade.
- `restricoes_linhas`, `restricoes_colunas`: as pistas de cada linha/coluna.
- `cnf`: lista de cláusulas que vai sendo construída (começa vazia).
- `var`: próximo número de variável auxiliar disponível. As primeiras `m * n` variáveis são reservadas para representar as células da grade (uma por célula), então as variáveis auxiliares de Tseitin começam a ser numeradas a partir de `m * n + 1`.
- `sol`: guarda a solução encontrada (ou `"UNSAT"`, ou `None` se o resolvedor ainda não foi executado).

In [53]:
def construtor_nonograma(self, nonograma: Nonograma):
    self.m = len(nonograma["linhas"])
    self.n = len(nonograma["colunas"])
    self.restricoes_linhas = nonograma["linhas"]
    self.restricoes_colunas = nonograma["colunas"]

    self.cnf = []
    self.var = self.m * self.n + 1  # próxima variável auxiliar
    self.sol = None

# Registra esse método como o construtor da classe ResolvedorNonograma
ResolvedorNonograma.__init__ = construtor_nonograma

### Escolhendo as variáveis

Introduzimos uma variável booleana por célula da grade:

$$ x_{ij} = \text{Verdadeiro se e somente se a célula na linha } i \text{, coluna } j \text{ estiver preenchida} $$

Para usá-las com um solucionador SAT, precisamos numerá-las de 1 a $m \times n$. Vamos numerar as células em ordem de leitura (linha a linha, da esquerda para a direita), de forma análoga à numeração de $x_{ijd}$ no Sudoku. A função `x(i, j)` abaixo devolve o número da variável correspondente à célula $(i, j)$.

In [54]:
def x(self, i: int, j: int) -> int:
    return i * self.n + j + 1

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.x = x

### Variáveis auxiliares para a transformação de Tseitin

Como veremos na próxima seção, para codificar "exatamente uma entre várias configurações possíveis" sem explodir o número de cláusulas, vamos precisar introduzir variáveis auxiliares — uma para cada configuração candidata de uma linha ou coluna.

O método `var_aux` simplesmente devolve um novo número de variável, sempre maior que o anterior, continuando a numeração logo após as $m \times n$ variáveis de célula.

In [55]:
def var_aux(self) -> int:
    v = self.var
    self.var += 1
    return v

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.var_aux = var_aux

### Enumerando as configurações válidas de uma linha/coluna

Dada uma restrição como `[2, 1]` para uma linha de tamanho `n`, existem várias formas de posicionar os blocos de comprimento 2 e 1 (nessa ordem, separados por ao menos uma célula vazia) dentro da linha. Cada uma dessas formas é uma *configuração* (ou permutação) válida — uma lista de `n` booleanos, `True` para célula preenchida.

A ideia para enumerar todas as configurações é distribuir a "folga" (espaço sobrando além do mínimo necessário) entre os `n_blocos + 1` "espaços" possíveis: antes do primeiro bloco, entre cada par de blocos consecutivos, e depois do último bloco. Se a folga for negativa (os blocos não cabem na linha, nem colados uns aos outros), a restrição é impossível de satisfazer e devolvemos `None`.

Casos especiais:

- Se a lista de restrições for vazia, a única configuração válida é a linha inteiramente vazia.
- Usamos `itertools.combinations_with_replacement` para gerar, de forma sistemática, todas as formas de distribuir a folga entre os espaços disponíveis.

In [56]:
from itertools import combinations_with_replacement
from typing import List

def permutacoes(self, n: int, restricoes: List[int]) -> List[List[bool]]:
    if not restricoes:
        return [[False] * n]

    n_blocos = len(restricoes)
    folga = n - sum(restricoes) - (n_blocos - 1)

    if folga < 0:
        return None

    n_baldes = n_blocos + 1  # número de "baldes" nos quais pode-se distribuir a folga

    permutacoes = []

    # gera combinações de onde cada folga pode cair entre os blocos
    for p in combinations_with_replacement(range(n_baldes), folga):
        qtd_baldes = [p.count(i) for i in range(n_baldes)]

        linha = []
        for i in range(n_blocos):
            linha.extend([False] * qtd_baldes[i])  # folga antes do bloco atual
            linha.extend([True] * restricoes[i])   # bloco atual
            if i < n_blocos - 1:
                linha.append(False)  # espaço entre os blocos

        # folga após o último bloco
        linha.extend([False] * qtd_baldes[-1])

        permutacoes.append(linha)
    return permutacoes

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.permutacoes = permutacoes

### Codificando "exatamente uma configuração" via Tseitin

Agora precisamos afirmar, em CNF, que uma linha (ou coluna) assume **uma das configurações válidas** enumeradas por `permutacoes`. Escrever isso diretamente como uma grande disjunção de conjunções (uma conjunção por configuração) exigiria distribuir OR sobre AND, gerando uma explosão exponencial de cláusulas.

Em vez disso, usamos a **transformação de Tseitin**: para cada configuração candidata $p$, criamos uma variável auxiliar $w_p$ que é logicamente equivalente a "a linha é exatamente igual à configuração $p$" ($w_p \leftrightarrow p$), onde $p$ é vista como a conjunção de um literal por célula — $x_j$ se a célula $j$ está preenchida em $p$, ou $\overline{x_j}$ se está vazia. Como as configurações são todas distintas entre si, basta então exigir que **pelo menos um** dos $w_p$ seja verdadeiro (a "raiz" da árvore de Tseitin) para garantir que exatamente uma configuração seja escolhida.

A equivalência $w_p \leftrightarrow p$ se decompõe em duas implicações, cada uma virando um conjunto diferente de cláusulas:

**1. $p \rightarrow w_p$ — uma única cláusula.**

Como $p = l_1 \land l_2 \land \dots \land l_n$ (um literal $l_j$ por célula), a implicação $p \rightarrow w_p$ equivale, por De Morgan, a

$$\overline{l_1} \lor \overline{l_2} \lor \dots \lor \overline{l_n} \lor w_p$$

ou seja, **uma cláusula só**, com $n+1$ literais: $w_p$ mais a negação de cada literal de $p$. É exatamente o que o laço interno monta em `p_entao_w`, acrescentando `-vars[j]` quando a célula $j$ está preenchida em $p$ (`p[j]` é `True`) e `vars[j]` quando está vazia — e essa lista só é adicionada a `self.cnf` **depois** do laço `for j in range(n)` terminar.

**2. $w_p \rightarrow p$ — uma cláusula por célula.**

Aqui a implicação vai para uma conjunção do lado de dentro: $w_p \rightarrow (l_1 \land l_2 \land \dots \land l_n)$ só é verdadeira se $w_p \rightarrow l_j$ valer para **cada** $j$ separadamente. Cada uma dessas implicações vira sua própria cláusula binária, $\overline{w_p} \lor l_j$, totalizando $n$ cláusulas — uma para cada célula da linha/coluna. É o que acontece dentro do próprio laço `for j in range(n)`: a cada iteração, `self.cnf.append([-w, vars[j]])` é a cláusula $\overline{w_p} \lor x_j$ (célula $j$ preenchida em $p$), e `self.cnf.append([-w, -vars[j]])` é a cláusula $\overline{w_p} \lor \overline{x_j}$ (célula $j$ vazia em $p$).

**Exemplo concreto.** Considere uma linha de tamanho $n=4$ com restrição `[2, 1]`, variáveis de célula $x_1, x_2, x_3, x_4$, e a configuração candidata $p = $ `[True, True, False, True]` (■■□■), com variável auxiliar $w_p = w$. As cláusulas geradas para essa configuração são:

- $p \rightarrow w$ (1 cláusula, `p_entao_w`): $(w \lor \overline{x_1} \lor \overline{x_2} \lor x_3 \lor \overline{x_4})$
- $w \rightarrow p$ (4 cláusulas, uma por célula): $(\overline{w} \lor x_1)$, $(\overline{w} \lor x_2)$, $(\overline{w} \lor \overline{x_3})$, $(\overline{w} \lor x_4)$

Ou seja, cada configuração $p$ contribui com $n + 1$ cláusulas (1 de $p \rightarrow w_p$ + $n$ de $w_p \rightarrow p$). Somando isso sobre todas as configurações válidas de uma linha/coluna, mais a cláusula "raiz" que exige pelo menos um $w_p$ verdadeiro, obtemos a CNF completa gerada por `gera_cnf_eixo`.

Dois casos especiais continuam valendo:

- Se a lista de restrições for vazia (`restricoes == []`), a única configuração válida é a linha vazia, então simplesmente forçamos todas as suas variáveis a serem falsas (sem precisar de variáveis auxiliares).
- Se não existir nenhuma configuração válida para a linha/coluna (`permutacoes` devolve `None`), a fórmula toda é insatisfatível e retornamos `False`.

In [57]:
def gera_cnf_eixo(self, restricoes: List[int], vars: List[int]) -> bool:
    # gera cnf para linha ou coluna

    n = len(vars)

    if restricoes == []:  # linha / coluna sem restrições
        for v in vars:
            self.cnf.append([-v])
        return True

    permutacoes = self.permutacoes(n, restricoes)
    if permutacoes is None:
        return False

    # variáveis auxiliares para a transformação de tseitin
    vars_aux = [self.var_aux() for _ in range(len(permutacoes))]
    self.cnf.append(vars_aux)  # raíz da árvore, uma das permutações deve ser verdadeira

    for i, p in enumerate(permutacoes):
        w = vars_aux[i]

        # p (permutação) -> w (cláusula auxiliar)
        # disjunção da negação das variáveis de p v w
        p_entao_w = [w]

        for j in range(n):
            if p[j]:
                p_entao_w.append(-vars[j])

                # w (cláusula auxiliar) -> p (permutação)
                self.cnf.append([-w, vars[j]])
            else:
                p_entao_w.append(vars[j])

                # w (cláusula auxiliar) -> p (permutação)
                self.cnf.append([-w, -vars[j]])

        self.cnf.append(p_entao_w)
    return True

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.gera_cnf_eixo = gera_cnf_eixo

### Colocando tudo junto: gerando a CNF completa

Com `gera_cnf_eixo` pronta para uma única linha ou coluna, `gera_cnf` simplesmente a chama para cada uma das `m` linhas e `n` colunas da grade, acumulando as cláusulas em `self.cnf`.

Se qualquer linha ou coluna for impossível de satisfazer isoladamente (por exemplo, blocos que não cabem no tamanho da grade), o Nonograma inteiro é insatisfatível: marcamos `self.sol = "UNSAT"`, limpamos o estado interno com `reset` e retornamos `False` imediatamente, sem gerar cláusulas desnecessárias para os eixos restantes.

In [58]:
def gera_cnf(self) -> bool:
    for i in range(self.m):
        vars_linha = [self.x(i, j) for j in range(self.n)]
        if not self.gera_cnf_eixo(self.restricoes_linhas[i], vars_linha):
            self.sol = "UNSAT"
            self.reset()
            return False

    for j in range(self.n):
        vars_coluna = [self.x(i, j) for i in range(self.m)]
        if not self.gera_cnf_eixo(self.restricoes_colunas[j], vars_coluna):
            self.sol = "UNSAT"
            self.reset()
            return False

    return True

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.gera_cnf = gera_cnf

### Resolvendo com o PycoSAT

`resolve` monta a CNF (via `gera_cnf`), passa a fórmula para `pycosat.solve` e trata os dois desfechos possíveis:

- Se a fórmula for insatisfatível, `pycosat.solve` devolve a string `"UNSAT"`, que guardamos em `self.sol`.
- Caso contrário, ele devolve a lista de todos os literais (positivos e negativos) atribuídos. Como só nos interessam as $m \times n$ primeiras variáveis (as células da grade — as variáveis auxiliares de Tseitin vêm depois e não fazem parte da solução), truncamos a lista, convertemos cada literal em um booleano (`True` se positivo) e reorganizamos em uma matriz `m` × `n`.

In [59]:
def resolve(self):
    if not self.gera_cnf():
        return

    output = pycosat.solve(self.cnf)
    if output == "UNSAT":
        self.sol = "UNSAT"
        return

    output_truncado = output[:self.m * self.n]
    output_bool = [True if var > 0 else False for var in output_truncado]
    self.sol = [output_bool[i : i + self.n] for i in range(0, len(output_bool), self.n)]

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.resolve = resolve

### Exibindo o resultado

Por fim, `resultado` imprime a grade resolvida de forma legível, junto com as pistas originais de linhas (à esquerda) e colunas (acima), alinhadas para formar um "tabuleiro" — células preenchidas são exibidas como `■` e vazias como `□`.

Se o resolvedor ainda não foi chamado, ou se o Nonograma for insatisfatível, `resultado` apenas informa isso em vez de tentar desenhar uma grade.

In [60]:
def resultado(self):
    if not self.sol:
        print("chame o resolvedor")
        return

    if isinstance(self.sol, str):
        print(self.sol)
        return

    max_linhas = max((len(r) for r in self.restricoes_linhas), default=0)
    max_colunas = max((len(c) for c in self.restricoes_colunas), default=0)

    # restrições colunas
    for i in range(max_colunas):
        linha_str = " " * (max_linhas * 2)  # recuo

        for j in range(self.n):
            restricoes = self.restricoes_colunas[j]
            offset = max_colunas - len(restricoes)

            if i < max_colunas - len(restricoes):
                linha_str += "  "
            else:
                linha_str += str(restricoes[i - offset]) + " "

        print(linha_str.rstrip())

    # restrições linhas + grade
    for i, row in enumerate(self.sol):
        restricoes = self.restricoes_linhas[i]

        restricoes_str = [str(x) for x in restricoes]
        padding = [" "] * (max_linhas - len(restricoes))
        linha_str = " ".join(padding + restricoes_str)

        linha_str = linha_str + " " + " ".join("■" if var else "□" for var in row)

        print(linha_str)

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.resultado = resultado

### Reiniciando o estado interno

`reset` limpa a lista de cláusulas, reinicia o contador de variáveis auxiliares (de volta para logo após as variáveis de célula) e apaga a solução guardada. É usado internamente por `gera_cnf` quando alguma linha/coluna já se mostra insatisfatível, mas também pode ser chamado manualmente para resolver o mesmo objeto `ResolvedorNonograma` novamente do zero.

In [61]:
def reset(self):
    self.cnf = []
    self.var = self.m * self.n + 1
    self.sol = None

# Registra esse método na classe ResolvedorNonograma
ResolvedorNonograma.reset = reset

## Testando o resolvedor

Vamos verificar se o nosso resolvedor consegue resolver corretamente o `nonograma_simples` definido no início do notebook.

In [62]:
resolvedor_simples = ResolvedorNonograma(nonograma_simples)

resolvedor_simples.resolve()
resolvedor_simples.resultado()

        1 1 1
      2 1 1 1 2
  1 1 □ ■ □ ■ □
1 1 1 ■ □ ■ □ ■
  1 1 ■ □ □ □ ■
  1 1 □ ■ □ ■ □
    1 □ □ ■ □ □


### Caso UNSAT

Nem todo conjunto de pistas corresponde a um Nonograma satisfatível. No exemplo abaixo, as três colunas exigem, respectivamente, nenhum bloco preenchido, um bloco de tamanho 3 e outro bloco de tamanho 3 — mas as três linhas exigem um bloco de tamanho 3 cada, o que é impossível de conciliar com a primeira coluna vazia. O resolvedor deve detectar corretamente que essa instância é insatisfatível.

In [63]:
nonograma_impossivel = {
    "linhas": [
        [3],
        [3],
        [3]
    ],
    "colunas": [
        [],
        [3],
        [3]
    ]
}

resolvedor_unsat = ResolvedorNonograma(nonograma_impossivel)
resolvedor_unsat.resolve()

resolvedor_unsat.resultado()

UNSAT


### Uma instância maior

Vamos testar um Nonograma 15×15 mais denso, para ter uma noção do tempo de execução em uma instância de tamanho mais realista.

In [64]:
import time

nonograma_grande = {
    "linhas": [
        [3], [4, 2, 1], [3, 1, 1, 1], [3, 1, 2, 1], [2, 1, 2], [4, 2, 2], [2, 2, 2], [1, 3, 1], [2, 2, 1], [4, 3], [1, 1, 2, 2], [1, 1, 1], [1, 1, 1], [5, 2], [3, 2, 3]
    ],
    "colunas": [
        [2], [3], [1, 1], [1, 1, 2, 1], [2, 2, 1, 1], [2, 6, 2], [1, 2, 2], [1, 3, 2], [3, 2, 5], [2, 1, 1], [1, 1], [4, 2, 1], [2, 2, 2, 2], [1, 1, 2, 5], [2, 5]
    ]
}

tempo_inicio = time.time()

resolvedor_grande = ResolvedorNonograma(nonograma_grande)
resolvedor_grande.resolve()

tempo_fim = time.time()
tempo_total = tempo_fim - tempo_inicio

resolvedor_grande.resultado()

print(f"[+] Tempo total gasto (Modelagem + SAT): {tempo_total:.4f} segundos.")

              1 2               2 1
              1 2 2 1 1 3 2   4 2 1
            1 2 1 6 2 3 2 1 1 2 2 2 2
        2 3 1 1 1 2 2 2 5 1 1 1 2 5 5
      3 □ □ □ □ □ □ □ □ □ □ □ □ ■ ■ ■
  4 2 1 □ □ □ □ □ ■ ■ ■ ■ □ □ ■ ■ □ ■
3 1 1 1 □ □ □ ■ ■ ■ □ □ ■ □ □ ■ □ ■ □
3 1 2 1 ■ ■ ■ □ ■ □ □ ■ ■ □ □ ■ □ □ □
  2 1 2 ■ ■ □ □ □ □ □ ■ □ □ □ ■ ■ □ □
  4 2 2 □ ■ ■ ■ ■ □ □ ■ ■ □ □ □ ■ ■ □
  2 2 2 □ □ □ □ ■ ■ □ □ ■ ■ □ □ □ ■ ■
  1 3 1 □ □ □ □ □ ■ □ □ □ ■ ■ ■ □ □ ■
  2 2 1 □ □ □ □ □ ■ ■ □ □ □ □ ■ ■ □ ■
    4 3 □ □ □ ■ ■ ■ ■ □ □ □ □ □ ■ ■ ■
1 1 2 2 □ □ □ ■ □ ■ □ □ ■ ■ □ □ □ ■ ■
  1 1 1 □ □ □ □ □ ■ □ □ ■ □ □ □ □ ■ □
  1 1 1 □ □ □ □ □ □ ■ □ ■ □ □ □ □ ■ □
    5 2 □ □ □ □ □ ■ ■ ■ ■ ■ □ □ ■ ■ □
  3 2 3 □ □ □ ■ ■ ■ □ ■ ■ □ ■ ■ ■ □ □
[+] Tempo total gasto (Modelagem + SAT): 0.2372 segundos.


### Um desafio maior: imagem 30×30

Como último teste de desempenho, vamos resolver um Nonograma 30×30 cujas pistas desenham um coração, para confirmar que o resolvedor lida bem com grades maiores em tempo aceitável.

In [65]:
import time

# IMAGEM 30x30 (coração)

nonograma_desafio = {
    "linhas": [
        [], [], [], [1, 1], [8, 8],
        [22], [24], [24], [26], [26],
        [26], [26], [26], [26], [26],
        [24], [24], [22], [22], [20],
        [18], [16], [14], [12], [10],
        [8], [4], [2], [], [],
    ],
    "colunas": [
        [], [], [7], [11], [14],
        [16], [17], [18], [19], [21],
        [21], [22], [22], [22], [23],
        [23], [22], [22], [22], [21],
        [21], [19], [18], [17], [16],
        [14], [11], [7], [], [],
    ]
}

tempo_inicio = time.time()

resolvedor_desafio = ResolvedorNonograma(nonograma_desafio)
resolvedor_desafio.resolve()

tempo_fim = time.time()
tempo_total = tempo_fim - tempo_inicio

resolvedor_desafio.resultado()

print(f"[+] Tempo total gasto: {tempo_total:.4f} segundos.")

        7 11 14 16 17 18 19 21 21 22 22 22 23 23 22 22 22 21 21 19 18 17 16 14 11 7
    □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □
    □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □
    □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □
1 1 □ □ □ □ □ □ □ □ □ ■ □ □ □ □ □ □ □ □ □ □ ■ □ □ □ □ □ □ □ □ □
8 8 □ □ □ □ □ ■ ■ ■ ■ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ ■ ■ ■ ■ □ □ □ □ □
  22 □ □ □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □ □ □
  24 □ □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □ □
  24 □ □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □ □
  26 □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □
  26 □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □
  26 □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □
  26 □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □
  26 □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □
  26 □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □
  26 □ □ ■ 

### Um padrão regular: tabuleiro de xadrez

Por fim, um Nonograma 20×20 com um padrão de blocos regulares (semelhante a um tabuleiro de xadrez), para observar o comportamento do resolvedor em instâncias com muita estrutura repetida.

In [66]:
import time

nonograma_xadrez = {
    "linhas": [
        [4, 4, 4], [4, 4, 4], [4, 4, 4], [4, 4, 4],
        [4, 4], [4, 4], [4, 4], [4, 4],
        [4, 4, 4], [4, 4, 4], [4, 4, 4], [4, 4, 4],
        [4, 4], [4, 4], [4, 4], [4, 4],
        [4, 4, 4], [4, 4, 4], [4, 4, 4], [4, 4, 4],
    ],
    "colunas": [
        [4, 4, 4], [4, 4, 4], [4, 4, 4], [4, 4, 4],
        [4, 4], [4, 4], [4, 4], [4, 4],
        [4, 4, 4], [4, 4, 4], [4, 4, 4], [4, 4, 4],
        [4, 4], [4, 4], [4, 4], [4, 4],
        [4, 4, 4], [4, 4, 4], [4, 4, 4], [4, 4, 4],
    ]
}

tempo_inicio = time.time()

resolvedor_xadrez = ResolvedorNonograma(nonograma_xadrez)
resolvedor_xadrez.resolve()

tempo_fim = time.time()
tempo_total = tempo_fim - tempo_inicio

resolvedor_xadrez.resultado()

print(f"[+] Tempo total gasto: {tempo_total:.4f} segundos.")

      4 4 4 4         4 4 4 4         4 4 4 4
      4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
      4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
  4 4 □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■
4 4 4 ■ ■ ■ ■ □ □ □ □ ■ ■ ■ ■ □ □ 

### Teste aleatório

Gerador aleatório de nonogramas válidos e inválidos, podendo ser alterado o tamanho e a densidade. Para esse puzzle, uma densidade menor significa ser mais dificil, já que significa uma maior explosão combinatória. Segundo os testes, o maior tamanho que nosso resolvedor consegue aguentar sem explodir a RAM é 24 (E leva ~160 segundos)

In [67]:
import random
import time

TAMANHO_GRADE = 19
densidade_escolhida = 0.5 # "FACIL = 0.85", "MEDIO = 0.5", "DIFICIL = 0.25"

def imprimir_puzzle_vazio(nonograma):
    restricoes_linhas = nonograma["linhas"]
    restricoes_colunas = nonograma["colunas"]
    m, n = len(restricoes_linhas), len(restricoes_colunas)

    max_linhas = max((len(r) for r in restricoes_linhas), default=0)
    max_colunas = max((len(c) for c in restricoes_colunas), default=0)

    for i in range(max_colunas):
        linha_str = " " * (max_linhas * 2)
        for j in range(n):
            restr = restricoes_colunas[j]
            offset = max_colunas - len(restr)
            if i < offset:
                linha_str += "  "
            else:
                linha_str += str(restr[i - offset]) + " "
        print(linha_str.rstrip())

    for i in range(m):
        restr = restricoes_linhas[i]
        restr_str = [str(x) for x in restr]
        padding = [" "] * (max_linhas - len(restr))
        linha_str = " ".join(padding + restr_str)
        linha_str += " " + " ".join("?" for _ in range(n))
        print(linha_str)
    print("\n")

def gerar_nonograma_aleatorio(linhas_qtd, colunas_qtd, densidade, forcar_unsat=False):
    grade = [[random.random() < densidade for _ in range(colunas_qtd)] for _ in range(linhas_qtd)]
    def extrair_pistas(vetor):
        pistas, contador = [], 0
        for celula in vetor:
            if celula: contador += 1
            elif contador > 0:
                pistas.append(contador)
                contador = 0
        if contador > 0: pistas.append(contador)
        return pistas

    restricoes_linhas = [extrair_pistas(grade[i]) for i in range(linhas_qtd)]
    restricoes_colunas = [extrair_pistas([grade[i][j] for i in range(linhas_qtd)]) for j in range(colunas_qtd)]

    if forcar_unsat and restricoes_linhas:
        if restricoes_linhas[0]: restricoes_linhas[0][0] += 1
        else: restricoes_linhas[0] = [1]

    return {"linhas": restricoes_linhas, "colunas": restricoes_colunas}

print(f" TESTE 1: MATRIZ {TAMANHO_GRADE}x{TAMANHO_GRADE} (SAT) - Densidade: {densidade_escolhida}")

instancia_sat = gerar_nonograma_aleatorio(TAMANHO_GRADE, TAMANHO_GRADE, densidade_escolhida, False)
print("O Problema Gerado:")
imprimir_puzzle_vazio(instancia_sat)

tempo_inicio = time.time()
resolvedor_sat = ResolvedorNonograma(instancia_sat)
resolvedor_sat.resolve()
tempo_total_sat = time.time() - tempo_inicio

print("A Solução do SAT Solver:")
resolvedor_sat.resultado()
print(f"\nTempo Gasto: {tempo_total_sat:.4f} segundos.\n")

print(f" TESTE 2: MATRIZ {TAMANHO_GRADE}x{TAMANHO_GRADE} (UNSAT) - Densidade: {densidade_escolhida}")

instancia_unsat = gerar_nonograma_aleatorio(TAMANHO_GRADE, TAMANHO_GRADE, densidade_escolhida, True)
print("Problema UNSAT:")
imprimir_puzzle_vazio(instancia_unsat)

tempo_inicio = time.time()
resolvedor_unsat = ResolvedorNonograma(instancia_unsat)
resolvedor_unsat.resolve()
tempo_total_unsat = time.time() - tempo_inicio

print("A Solução do SAT Solver:")
resolvedor_unsat.resultado()
print(f"\nTempo Gasto: {tempo_total_unsat:.4f} segundos.")

 TESTE 1: MATRIZ 19x19 (SAT) - Densidade: 0.5
O Problema Gerado:
                                            1
                1   3   1 1   2 1       1   2       1
                2 2 4   2 2 1 1 1 1 1 1 1   1   4   1
                1 1 1 3 2 3 3 2 4 3 1 1 3 1 1 1 1   1
                1 1 1 3 1 1 2 1 2 2 2 1 2 4 2 8 1 2 1
                1 5 1 1 3 1 4 1 1 1 3 1 1 4 1 1 1 3 6
                1 1 1 3 1 1 2 2 1 3 1 2 1 1 1 3 4 4 1
    4 1 1 1 1 2 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
      4 1 1 2 2 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
          2 3 2 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
    2 1 5 1 1 1 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
      1 4 1 3 1 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
1 1 1 1 1 1 1 2 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
        3 1 3 1 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
    1 1 3 2 1 4 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
      1 1 2 2 2 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
    3 4 1 1 1 1 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ?
        1 1 7 4 ? ? ? ? ?

### Web Scaping

Um leitor simples de arquivo .non e .ss para que possamos buscar problemas na internet a partir da base de dados do WebPBN

In [68]:
def carregar_formato_non(texto_non: str, forcar_unsat: bool = False):
    """
    Lê o texto de um arquivo .non ou .ss e converte para o formato de dicionário.
    Ignora automaticamente qualquer metadado (textos, títulos, cabeçalhos)
    que não seja número de fato.
    """
    linhas = []
    colunas = []

    modo_leitura = None

    for linha in texto_non.strip().split('\n'):
        linha = linha.strip()

        if not linha:
            continue

        if linha.lower() == 'rows':
            modo_leitura = 'linhas'
            continue
        elif linha.lower() == 'columns':
            modo_leitura = 'colunas'
            continue

        if modo_leitura is None:
            continue

        try:
            pistas = [int(x) for x in linha.split(',')] if ',' in linha else [int(x) for x in linha.split()]

            if pistas == [0]:
                pistas = []

            if modo_leitura == 'linhas':
                linhas.append(pistas)
            elif modo_leitura == 'colunas':
                colunas.append(pistas)

        except ValueError:
            continue

    if forcar_unsat and linhas:
        if linhas[0]: linhas[0][0] += 1
        else: linhas[0] = [1]

    return {
        "linhas": linhas,
        "colunas": colunas
    }

### Testando em problemas reais

Para modificar o problema, mude o ID em "ID_ALVO" para desenhos diferentes

In [69]:
import urllib.request
import time

ID_ALVO = 105

def buscar_webpbn(id_puzzle: int, forcar_unsat: bool = False):
    url = f"https://webpbn.com/export.cgi?fmt=ss&id={id_puzzle}&go=1"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'
    }

    print(f"Baixando puzzle #{id_puzzle} do WebPBN...")
    try:
        req = urllib.request.Request(url, headers=headers)
        resposta = urllib.request.urlopen(req)

        texto_non = resposta.read().decode('utf-8')

        if not texto_non.strip():
            print("Arquivo vazio")
            return None

        if texto_non.strip().startswith('<!DOCTYPE') or texto_non.strip().startswith('<html'):
            print("O servidor recusou a exportação")
            return None

        print("Arquivo baixado com sucesso")
        return carregar_formato_non(texto_non, forcar_unsat)

    except Exception as erro:
        print(f"Falha na conexão de rede: {erro}")
        return None

# TESTE
instancia_web = buscar_webpbn(ID_ALVO, forcar_unsat=False)

if instancia_web:
    print(f"Resolvendo puzzle #{ID_ALVO}...")

    tempo_inicio = time.time()
    resolvedor_web = ResolvedorNonograma(instancia_web)
    resolvedor_web.resolve()
    tempo_fim = time.time()

    resolvedor_web.resultado()
    print(f"\nTempo total gasto: {tempo_fim - tempo_inicio:.4f} segundos.")

Baixando puzzle #105 do WebPBN...
Arquivo baixado com sucesso
Resolvendo puzzle #105...
                                1
                                1
                              1 8               2 1         2   3 4
                  6 5 4 3 2   1 1         16   11 8 10 2     1 3 2 1 2 5 6
              10 8 3 3 3 4 3 2 5 2   23 20 18 1 14 4 3 1 8 1 1 6 1 2 1 1 3 2 8
              7 5 4 3 2 1 1 4 1 1 26 2 3 6 4 5 2 4 1 6 18 17 8 5 8 2 3 4 5 7
          10 10 ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ □ □ □ □ □ □ □ □ □ □ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■
          8 7 ■ ■ ■ ■ ■ ■ ■ ■ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ ■ ■ ■ ■ ■ ■ ■
          6 5 ■ ■ ■ ■ ■ ■ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ □ ■ ■ ■ ■ ■
        5 1 4 ■ ■ ■ ■ ■ □ □ □ □ □ □ ■ □ □ □ □ □ □ □ □ □ □ □ □ □ □ ■ ■ ■ ■
        4 3 3 ■ ■ ■ ■ □ □ □ □ □ □ ■ ■ ■ □ □ □ □ □ □ □ □ □ □ □ □ □ □ ■ ■ ■
        3 6 2 ■ ■ ■ □ □ □ □ □ ■ ■ ■ ■ ■ ■ □ □ □ □ □ □ □ □ □ □ □ □ □ □ ■ ■
        2 5 1 ■ ■ □ □ □ □ □ □ □ □ ■ ■ ■ ■ ■ □ □ □ □ □ □ □ □ □ □ □ □ □ □ ■
        2 5 1 ■ ■ □ □ □ □